# Pick Simulation
Three pickers working through a batch across a 24-aisle warehouse.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'Warehouse'))

In [ ]:
import random
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from Warehouse_Builder import Warehouse_Builder, WarehouseConfig, AisleConfig
from Inventory_Builder import Inventory_Builder, InventoryConfig
from Inventory_Management import Inventory_Manager
from Workload_Builder import BatchConfig, Batch, Task
from Pick import PickConfig, PickSimulation

## Warehouse & Inventory

In [ ]:
config = WarehouseConfig(
    total_aisles=24,
    aisle_splits=[1/24] * 24,
    aisle_configs=[
        AisleConfig(handling_type='conveyable',     storage_type='food',       unit_type='pallet',    bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='food',       unit_type='singleton', bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='clothing',   unit_type='pallet',    bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='clothing',   unit_type='singleton', bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='electronic', unit_type='pallet',    bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='electronic', unit_type='singleton', bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='furniture',  unit_type='pallet',    bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='furniture',  unit_type='singleton', bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='seasonal',   unit_type='pallet',    bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='seasonal',   unit_type='singleton', bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='chemical',   unit_type='pallet',    bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='conveyable',     storage_type='chemical',   unit_type='singleton', bayXPerAisle=5, bayYPerAisle=6, storage_sizes=['medium']),
        AisleConfig(handling_type='non-conveyable', storage_type='food',       unit_type='pallet',    bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='food',       unit_type='singleton', bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='clothing',   unit_type='pallet',    bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='clothing',   unit_type='singleton', bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='electronic', unit_type='pallet',    bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='electronic', unit_type='singleton', bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='furniture',  unit_type='pallet',    bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='furniture',  unit_type='singleton', bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='seasonal',   unit_type='pallet',    bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='seasonal',   unit_type='singleton', bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='chemical',   unit_type='pallet',    bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
        AisleConfig(handling_type='non-conveyable', storage_type='chemical',   unit_type='singleton', bayXPerAisle=4, bayYPerAisle=6, storage_sizes=['large']),
    ]
)
warehouse = Warehouse_Builder().from_config(config).build()
print(f'Aisles     : {len(warehouse.aisles)}')
print(f'Total bins : {len(warehouse.bins)}')

In [ ]:
inv_config = InventoryConfig(
    num_skus=120,
    handling_splits=[0.5, 0.5],
    category_splits=[1/6] * 6,
)
inventory = Inventory_Builder().from_config(inv_config).build()

manager = Inventory_Manager(warehouse)
manager.enqueue_all(inventory.cartons, quantity=5)
manager.summary()

## Batch & Tasks

In [ ]:
random.seed(42)
affinity      = inventory.affinity_matrix()
batch_cfg     = BatchConfig(inventory_size=len(inventory.cartons), mean_fraction=0.30, std_fraction=0.05)
batch         = Batch(batch_cfg, inventory, affinity=affinity)
tasks         = Task.from_batch(batch, warehouse)

print(f'Batch SKUs : {len(batch.items)}')
print(f'Tasks      : {len(tasks)}')
print(f'Aisles hit : {sorted(t.aisle_id for t in tasks)}')

In [ ]:
pd.DataFrame([{
    'aisle_id'      : t.aisle_id,
    'picks'         : len(t.items),
    'stops'         : len(t.path),
    'x_traversed'   : t.x_traversed,
    'y_traversed'   : t.y_traversed,
    'carts_required': t.carts_required,
} for t in tasks]).sort_values('aisle_id').reset_index(drop=True)

## Pick Simulation — 3 Pickers

In [ ]:
pick_cfg = PickConfig(
    num_pickers      = 3,
    x_move_time      = 1.0,
    y_move_time      = 0.5,
    pick_intercept   = 1.0,
    pick_weight_coef = 0.1,
    pick_volume_coef = 1e-4,
    cart_swap_coef   = 5.0,
)

sim    = PickSimulation(tasks, pick_cfg)
events = sim.run()

finish_times = {pid: max(e.time for e in events if e.picker_id == pid) for pid in range(3)}
print(f'Total events : {len(events)}')
for pid, t in finish_times.items():
    print(f'  Picker {pid} finishes at t={t:.2f}')
print(f'Makespan     : {max(finish_times.values()):.2f} time units')

## Event Log

In [ ]:
df_events = pd.DataFrame([{
    'time'    : round(e.time, 2),
    'picker'  : e.picker_id,
    'event'   : e.event_type,
    'aisle'   : e.aisle_id,
    'sku'     : e.sku,
    'qty'     : e.quantity,
    'bins'    : f"{e.bins_completed}/{e.total_bins}" if e.total_bins else '',
    'items'   : e.items_picked,
} for e in events])
df_events

## Progress at Each Time Step

In [ ]:
table = sim.step_table(step=10.0)

rows = []
for snapshots in table:
    for p in snapshots:
        rows.append({
            't'          : p.time,
            'picker'     : p.picker_id,
            'status'     : p.status,
            'aisle'      : p.task_aisle_id,
            'bins_done'  : p.bins_completed,
            'total_bins' : p.total_bins,
            'items'      : p.items_picked,
            'carts'      : p.carts_used,
            'progress'   : f'{p.progress:.0%}',
        })

pd.DataFrame(rows)

## Activity Timeline (Gantt)

In [ ]:
_STATUS_COLORS = {
    'traveling': '#aec6cf',
    'picking'  : '#ffad60',
    'cart_swap': '#ff6b6b',
    'idle'     : '#e0e0e0',
}
_EVENT_STATUS = {
    'task_start': 'traveling',
    'arrive'    : 'picking',
    'cart_swap' : 'cart_swap',
    'pick'      : 'traveling',
    'task_end'  : 'traveling',
    'done'      : 'idle',
}

fig, axes = plt.subplots(3, 1, figsize=(15, 6), sharex=True)

for pid, ax in enumerate(axes):
    pevents = [e for e in events if e.picker_id == pid]
    for i in range(len(pevents) - 1):
        e0, e1 = pevents[i], pevents[i + 1]
        status = _EVENT_STATUS.get(e0.event_type, 'idle')
        ax.barh(0, e1.time - e0.time, left=e0.time, height=0.6,
                color=_STATUS_COLORS[status], edgecolor='none')
        if e0.event_type == 'task_start' and e0.aisle_id is not None:
            ax.text(e0.time + 0.3, 0, f'A{e0.aisle_id}',
                    va='center', fontsize=7, color='#333')
    ax.set_yticks([])
    ax.set_ylabel(f'Picker {pid}', rotation=0, labelpad=45, va='center')

axes[-1].set_xlabel('Time units')
fig.suptitle('Picker Activity Timeline', fontsize=13, fontweight='bold', y=1.01)

legend_patches = [mpatches.Patch(color=c, label=l) for l, c in _STATUS_COLORS.items()]
axes[0].legend(handles=legend_patches, loc='upper right', fontsize=8, framealpha=0.9)
plt.tight_layout()
plt.show()

## Progress Over Time

In [ ]:
table_fine = sim.step_table(step=5.0)
_COLORS = ['#2196F3', '#FF5722', '#4CAF50']

fig, ax = plt.subplots(figsize=(13, 4))
for pid in range(3):
    times    = [s[pid].time     for s in table_fine]
    progress = [s[pid].progress for s in table_fine]
    ax.plot(times, progress, marker='.', markersize=4,
            label=f'Picker {pid}', color=_COLORS[pid])

ax.set_xlabel('Time units')
ax.set_ylabel('Task progress')
ax.set_title('Picker Progress Over Time')
ax.set_ylim(-0.05, 1.1)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.legend()
plt.tight_layout()
plt.show()